# Function 5: Part 2 analysis notebook

This notebook keeps the original data-loading cells, appends the latest query point/output from the end of your uploaded notebook, and then runs one focused analysis for Part 2.

Assumption: lower output is better, so the optimisation target is minimisation.


In [1]:
import numpy as np

input_data = np.load('../../data/initial_data/function_5/initial_inputs.npy')
print("Before:", input_data.shape)
new_point = np.array([
    [0.278167, 0.217734, 0.996929, 0.992772],
    [0.331025, 0.592662, 0.991036, 0.994402],
    [0.5, 0.5, 0.5, 0.5]
    ])
input_data = np.vstack([input_data, new_point])
print("After:", input_data.shape)
print(input_data)


Before: (20, 4)
After: (23, 4)
[[0.19144708 0.03819337 0.60741781 0.41458414]
 [0.75865295 0.53651774 0.65600038 0.36034155]
 [0.43834987 0.8043397  0.21024527 0.15129482]
 [0.70605083 0.53419196 0.26424335 0.48208755]
 [0.83647799 0.19360965 0.6638927  0.78564888]
 [0.68343225 0.11866264 0.82904591 0.56757661]
 [0.55362148 0.66734998 0.32380582 0.81486975]
 [0.35235627 0.32224153 0.11697937 0.47311252]
 [0.15378571 0.72938169 0.42259844 0.44307417]
 [0.46344227 0.63002451 0.10790646 0.9576439 ]
 [0.67749115 0.35850951 0.47959222 0.07288048]
 [0.58397341 0.14724265 0.34809746 0.42861465]
 [0.30688872 0.31687813 0.62263448 0.09539906]
 [0.51114177 0.817957   0.72871042 0.11235362]
 [0.43893338 0.77409176 0.37816709 0.93369621]
 [0.22418902 0.84648049 0.87948418 0.87851568]
 [0.72526172 0.47987049 0.08894684 0.75976022]
 [0.35548161 0.63961937 0.41761768 0.12260384]
 [0.11987923 0.86254031 0.64333133 0.84980383]
 [0.12688467 0.15342962 0.77016219 0.19051811]
 [0.278167   0.217734   0.996

In [ ]:
output_data = np.load('../../data/initial_data/function_5/initial_outputs.npy')
print("Before:", output_data.shape)
new_output = np.array([
    1552.6699801013826,
    1803.9236591562037,
    -0.015979341188442648
    ])
output_data = np.append(output_data, new_output)
print("After:", output_data.shape)
print(output_data)


Before: (20,)
After: (22,)
[6.44434399e+01 1.83013796e+01 1.12939795e-01 4.21089813e+00
 2.58370525e+02 7.84343889e+01 5.75715369e+01 1.09571876e+02
 8.84799176e+00 2.33223610e+02 2.44230883e+01 6.44201468e+01
 6.34767158e+01 7.97291299e+01 3.55806818e+02 1.08885962e+03
 2.88667516e+01 4.51815703e+01 4.31612757e+02 9.97233189e+00
 1.55266998e+03 1.80392366e+03]


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Latest query/output extracted from the uploaded notebook.
# Edit these if you later receive a different portal output.
latest_query = np.array([[0.5, 0.5, 0.5, 0.5]])
actual_output = 32.0025

# Append the latest query/output only if it is not already present.
if actual_output is not None:
    already_present = np.any(np.all(np.isclose(input_data, latest_query, atol=1e-12), axis=1))
    if not already_present:
        input_data = np.vstack([input_data, latest_query])
        output_data = np.append(output_data, actual_output)

function_id = 5
d = input_data.shape[1]
print(f"Function {function_id}, dimension d={d}")
print("Data shape:", input_data.shape, output_data.shape)
print("Current best observed y:", output_data.min())
print("Current best x:", input_data[np.argmin(output_data)])


Function 5, dimension d=4
Data shape: (23, 4) (23,)
Current best observed y: 0.1129397953712203
Current best x: [0.43834987 0.8043397  0.21024527 0.15129482]


In [4]:
# Basic table used in all interpretations
summary = pd.DataFrame(input_data, columns=[f"x{i+1}" for i in range(d)])
summary["y"] = output_data
summary["log_abs_y"] = np.log(np.abs(output_data) + 1e-300)
summary["rank_min"] = summary["y"].rank(method="first", ascending=True).astype(int)
summary = summary.sort_values("y")
display(summary)


,x1,x2,x3,x4,y,log_abs_y,rank_min
2,0.438350,0.804340,0.210245,0.151295,0.112940,-2.180900,1
3,0.706051,0.534192,0.264243,0.482088,4.210898,1.437676,2
8,0.153786,0.729382,0.422598,0.443074,8.847992,2.180191,3
19,0.126885,0.153430,0.770162,0.190518,9.972332,2.299814,4
1,0.758653,0.536518,0.656000,0.360342,18.301380,2.906976,5
10,0.677491,0.358510,0.479592,0.072880,24.423088,3.195529,6
16,0.725262,0.479870,0.088947,0.759760,28.866752,3.362690,7
22,0.500000,0.500000,0.500000,0.500000,32.002500,3.465814,8
17,0.355482,0.639619,0.417618,0.122604,45.181570,3.810689,9
6,0.553621,0.667350,0.323806,0.814870,57.571537,4.053028,10


## Gaussian process surrogate + expected improvement for minimisation

In [5]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.preprocessing import StandardScaler
from scipy.stats import norm

# Standardise inputs and outputs for numerical stability.
X = input_data.copy()
y = output_data.copy()

x_scaler = StandardScaler()
y_scaler = StandardScaler()
X_scaled = x_scaler.fit_transform(X)
y_scaled = y_scaler.fit_transform(y.reshape(-1, 1)).ravel()

kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=np.ones(d), length_scale_bounds=(1e-2, 1e2), nu=2.5) + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-10, 1e-1))

gp = GaussianProcessRegressor(kernel=kernel, normalize_y=False, n_restarts_optimizer=20, random_state=0)
gp.fit(X_scaled, y_scaled)

print("Fitted kernel:", gp.kernel_)
print("Best observed y:", y.min())
print("Best observed x:", X[np.argmin(y)])


Fitted kernel: 3.08**2 * Matern(length_scale=[100, 14.5, 5.76, 3.03], nu=2.5) + WhiteKernel(noise_level=1e-10)
Best observed y: 0.1129397953712203
Best observed x: [0.43834987 0.8043397  0.21024527 0.15129482]


/Users/andriy/miniconda3/envs/appenv/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:430: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k2__length_scale is close to the specified upper bound 100.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
/Users/andriy/miniconda3/envs/appenv/lib/python3.11/site-packages/sklearn/gaussian_process/kernels.py:420: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-10. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [6]:
def expected_improvement_min(X_candidates, gp, y_best_scaled, xi=0.01):
    """Expected improvement for minimisation in scaled y-space."""
    mu, sigma = gp.predict(X_candidates, return_std=True)
    sigma = np.maximum(sigma, 1e-12)
    improvement = y_best_scaled - mu - xi
    Z = improvement / sigma
    ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
    return ei

# Random candidate search in [0, 1]^d. Increase n_candidates for a more exhaustive search.
rng = np.random.default_rng(0)
n_candidates = 20000 if d <= 4 else 50000
candidates = rng.random((n_candidates, d))
candidates_scaled = x_scaler.transform(candidates)

y_best_scaled = y_scaled.min()
ei = expected_improvement_min(candidates_scaled, gp, y_best_scaled, xi=0.01)
mu_scaled, std_scaled = gp.predict(candidates_scaled, return_std=True)
mu = y_scaler.inverse_transform(mu_scaled.reshape(-1, 1)).ravel()
std = std_scaled * y_scaler.scale_[0]

results = pd.DataFrame(candidates, columns=[f"x{i+1}" for i in range(d)])
results["pred_mean"] = mu
results["pred_std"] = std
results["expected_improvement"] = ei
results = results.sort_values("expected_improvement", ascending=False)

display(results.head(10))

best_next = results.iloc[0][[f"x{i+1}" for i in range(d)]].to_numpy(dtype=float)
print("Suggested next query:", np.round(best_next, 6))
print("Portal format:", ", ".join(f"x{i+1}={v:.6f}" for i, v in enumerate(best_next)))


,x1,x2,x3,x4,pred_mean,pred_std,expected_improvement
15166,0.785714,0.991121,0.255305,0.637310,-146.242005,83.837408,0.291400
8354,0.849861,0.983731,0.196884,0.639512,-143.751215,90.751588,0.288148
17543,0.973792,0.988486,0.180507,0.581541,-141.390039,89.527548,0.283332
14619,0.659622,0.981728,0.236301,0.672286,-136.601377,83.933047,0.272796
11939,0.529781,0.998814,0.286079,0.603133,-135.888668,75.883992,0.269727
4468,0.948744,0.947749,0.252701,0.583818,-135.587828,73.298310,0.268688
17818,0.895027,0.995480,0.147163,0.568919,-131.754712,94.130522,0.266526
18890,0.674978,0.967784,0.195416,0.633083,-133.070973,85.149920,0.266353
2539,0.455780,0.986128,0.272943,0.629820,-133.386650,76.597341,0.264991
2682,0.928070,0.965731,0.226542,0.559611,-133.444379,74.830131,0.264764


Suggested next query: [0.785714 0.991121 0.255305 0.63731 ]
Portal format: x1=0.785714, x2=0.991121, x3=0.255305, x4=0.637310


In [7]:
# Optional 2D visualisation only when d=2
if d == 2:
    grid_res = 150
    xx, yy = np.meshgrid(np.linspace(0, 1, grid_res), np.linspace(0, 1, grid_res))
    grid = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = x_scaler.transform(grid)
    mu_grid_scaled, std_grid_scaled = gp.predict(grid_scaled, return_std=True)
    mu_grid = y_scaler.inverse_transform(mu_grid_scaled.reshape(-1, 1)).reshape(grid_res, grid_res)
    ei_grid = expected_improvement_min(grid_scaled, gp, y_best_scaled).reshape(grid_res, grid_res)

    plt.figure(figsize=(7, 6))
    cf = plt.contourf(xx, yy, mu_grid, levels=40)
    plt.colorbar(cf, label="GP predicted mean")
    plt.scatter(input_data[:, 0], input_data[:, 1], c=output_data, edgecolors="black", s=80)
    plt.scatter(best_next[0], best_next[1], marker="*", s=250, edgecolors="black", label="suggested next")
    plt.xlabel("x1"); plt.ylabel("x2"); plt.title("GP predicted mean")
    plt.legend(); plt.show()

    plt.figure(figsize=(7, 6))
    cf = plt.contourf(xx, yy, ei_grid, levels=40)
    plt.colorbar(cf, label="Expected improvement")
    plt.scatter(input_data[:, 0], input_data[:, 1], c="white", edgecolors="black", s=80)
    plt.scatter(best_next[0], best_next[1], marker="*", s=250, edgecolors="black", label="suggested next")
    plt.xlabel("x1"); plt.ylabel("x2"); plt.title("Expected improvement for minimisation")
    plt.legend(); plt.show()
